In [141]:
import pandas as pd 
import sqlite3
import matplotlib.pyplot as plt # For plotting 
import seaborn as sns 
import plotly.express as px # For plotting 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path 

PROJECT_ROOT = Path.cwd().parent

DATA = 'covid.db'

conn = sqlite3.connect(DATA)

In [142]:
# Formatting SQL Datetime correctly

temp_df = pd.read_csv(PROJECT_ROOT/'data'/'raw'/'worldometer_coronavirus_daily_data.csv')

temp_df.dtypes

date                        object
country                     object
cumulative_total_cases     float64
daily_new_cases            float64
active_cases               float64
cumulative_total_deaths    float64
daily_new_deaths           float64
dtype: object

In [143]:
temp_df['date'] = pd.to_datetime(temp_df['date']).dt.strftime("%Y-%m-%d")

temp_df.to_sql(name='covid_cases', con=conn, if_exists='replace', index=False)

184787

In [144]:
query = """
SELECT 
    country, 
    date,
    SUM(daily_new_cases) AS total_cases, 
    SUM(daily_new_deaths) AS total_deaths,
    MIN(date(date)) AS count_begin,
    MAX(date(date)) AS count_end
FROM covid_cases
GROUP BY country  

"""

df = pd.read_sql(query, conn)

# Plot of total_cases for each countries 

fig = px.bar(df.sort_values('total_cases', ascending=False).dropna(), 
             x='country', y='total_cases',
             hover_data=['total_cases'])
fig.update_layout(
    xaxis=dict(tickangle=90, tickfont=dict(size=7)),
    height=600,
    width=1400,
    title="Total COVID Cases by Country"
)

fig.write_html(PROJECT_ROOT/ "figures" / "Total_COVID_by_Country.html")

fig.show()

In [145]:
df[df['total_cases'] == df['total_cases'].max()]

,country,date,total_cases,total_deaths,count_begin,count_end
212,USA,2022-05-14,84209458.0,1026646.0,2020-02-15,2022-05-14


Between 2020-02-15 and 2022-05-14, USA has total of around 84 million cases, with 1.03 million deaths. This is the highest among all other countries.

In [146]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country       226 non-null    object 
 1   date          226 non-null    object 
 2   total_cases   226 non-null    float64
 3   total_deaths  215 non-null    float64
 4   count_begin   226 non-null    object 
 5   count_end     226 non-null    object 
dtypes: float64(2), object(4)
memory usage: 10.7+ KB


In [147]:
df[df['count_begin'] == '2020-02-15'] # All countries except China begins at this date 

,country,date,total_cases,total_deaths,count_begin,count_end
0,Afghanistan,2022-05-14,179242.0,7690.0,2020-02-15,2022-05-14
1,Albania,2022-05-14,275572.0,3496.0,2020-02-15,2022-05-14
2,Algeria,2022-05-14,265815.0,6875.0,2020-02-15,2022-05-14
3,Andorra,2022-05-14,42155.0,153.0,2020-02-15,2022-05-14
4,Angola,2022-05-14,99193.0,1898.0,2020-02-15,2022-05-14
...,...,...,...,...,...,...
221,Wallis And Futuna Islands,2022-05-14,453.0,6.0,2020-02-15,2022-05-14
222,Western Sahara,2022-05-14,6.0,NaN,2020-02-15,2022-05-14
223,Yemen,2022-05-14,11818.0,2147.0,2020-02-15,2022-05-14
224,Zambia,2022-05-14,320589.0,3982.0,2020-02-15,2022-05-14


In [148]:
# Plot of USA Daily New Cases and Daily Death Rate 

sns.set_theme(
    style="whitegrid",  
    context="notebook",  
    palette="viridis",   
    font="Arial",        
    font_scale=1.2       
)

# ------------- SQL query -----------------
query = """
SELECT date, daily_new_cases, daily_new_deaths
FROM covid_cases 
WHERE country == 'USA'
"""

df_plot = pd.read_sql(query, conn)

df_plot['date'] = pd.to_datetime(df_plot['date'])

# ------------ Daily New Cases -------------

df_plot['cases_7d'] = df_plot['daily_new_cases'].rolling(7).mean()

peak_idx = df_plot['cases_7d'].idxmax()

peak_date = df_plot.loc[peak_idx, 'date']
peak_case = df_plot.loc[peak_idx, 'cases_7d']

# ------------ Daily Death Rate -------------

df_plot['Daily_death_rate(%)'] = ( df_plot['daily_new_deaths'] / df_plot['daily_new_cases'] ) * 100 
df_plot['Daily_death_rate7d'] = df_plot['Daily_death_rate(%)'].rolling(7).mean()

peak_death_idx = df_plot['Daily_death_rate7d'].idxmax()

peak_death = df_plot.loc[peak_death_idx, 'Daily_death_rate7d']
peak_death_date = df_plot.loc[peak_death_idx, 'date']
# ------------ Plotting --------------------
fig = make_subplots(rows=1, cols=2, subplot_titles=("Daily Cases", "Daily Death Rate"), column_widths=[0.7, 0.7])

fig.add_trace(go.Scatter( 
              x=df_plot['date'],
              y=df_plot['cases_7d'], 
              mode='lines',
              name='Daily new cases (USA)',
              line=dict(color='#1f77b4')),
              row=1, col=1)

fig.add_trace(go.Scatter( 
              x=df_plot['date'], 
              y=df_plot['Daily_death_rate7d'], 
              mode='lines',
              name='Daily Death Rate (USA)', 
              line=dict(color='#e377c2')),
              row=1, col=2)

fig.add_trace(go.Scatter(x=[peak_date], y=[peak_case], 
                         mode='markers', 
                         marker=dict(color='purple', size=8, symbol='star'),
                         name=f"Peak Case: ({peak_date.strftime('%Y-%m-%d')}, {round(peak_case)})"),
             row=1, col=1)

fig.add_trace(go.Scatter(x=[peak_death_date], y=[peak_death], 
                         mode='markers', 
                         marker=dict(color='black', size=8, symbol='star'),
                         name=f"Peak Death rate (%): ({peak_death_date.strftime('%Y-%m-%d')}, {round(peak_death, 3)}%)"),
              row=1, col=2)

fig.update_xaxes(title_text="Date", row=1, col=1)
fig.update_yaxes(title_text="Cases", row=1, col=1)

fig.update_xaxes(title_text="Date", row=1, col=2)
fig.update_yaxes(title_text="Death Rate", row=1, col=2)

fig.update_layout(
    width=1500,
    height=500
)
fig.write_html(PROJECT_ROOT/ "figures" / "covid_daily_cases_deaths_USA.html")

fig.show()

From this plot we observe that intitially (2020-02 - 2022-2), the daily COVID-cases remained stable however between 2022-01 and 2022-04 the daily new cases spiked with peak 824151 cases. 

Initially, the death rate was quiet high, however there were not much people is diagnosed at this point. The first death reported to be March 3rd, 2020.

## Total death rate for listed countries

In [149]:
# Calculating Total Death rate for each country 
df['Death_Rate'] = (df['total_deaths'] / df['total_cases']) * 100 

fig = px.bar(df.sort_values('Death_Rate', ascending=False).dropna(), 
       x='country', 
       y='Death_Rate',
       hover_data=['Death_Rate'],
       color_discrete_sequence=['orange'])
    
    
fig.update_layout(
    xaxis=dict(tickangle=90, tickfont=dict(size=7)),
    height=600,
    width=1400,
    title="Total Death Rate By Country"
)

fig.write_html(PROJECT_ROOT/ "figures" / "Total_Death_Rate_By_Country.html")

fig.show()

In [218]:
# Pie Chart 

# Sum up the total deaths and total cases 

total_cases = df['total_cases'].sum()
total_deaths = df['total_deaths'].sum()

# Sorting the df 

df_case_sorted = df.sort_values('total_cases', ascending=False)
df_death_sorted = df.sort_values('total_deaths', ascending=False)


df_case_sorted.loc[df_case_sorted.index[20:], 'country'] = 'Other countries'
df_death_sorted.loc[df_death_sorted.index[20:], 'country'] = 'Other countries'

# ---------------- Plot ----------------
fig = make_subplots(rows=1, cols=2, 
                    specs=[[{'type': 'domain'}, {'type': 'domain'}]],
                    subplot_titles=("Total Cases ", "Total Deaths"))

fig.add_trace(go.Pie(values=df_case_sorted['total_cases'],
                     labels=df_case_sorted['country'],
                     name='Cases',
                     scalegroup='one'
                    ),
                     1, 1)

fig.add_trace(go.Pie(values=df_death_sorted['total_deaths'],
                     labels=df_death_sorted['country'],
                     scalegroup='two',
                     name='Deaths',
                    ),
                     1, 2)
# fig = px.pie(df_case_sorted, 
#              values='total_cases', 
#              names='country', 
#              title='Total Case Distribution')

fig.update_traces(textinfo='percent+label')
fig.update_layout(width=1200, 
                  height=800,
                  legend=dict(
                  x=1.1,  # Move right (outside chart)
                  y=0.5),
                  margin=dict(r=150),
                  title='Distributions By Country')


fig.write_html(PROJECT_ROOT/ "figures" / "Cases_Deaths_Distribution_by_Country.html")

fig.show()

In [10]:
conn.close()